Run all cells in order: the plots depend on the complete sequence of random draws, including the draws of the cells that only display intermediate results.

The dataset class is adapted from the [UvA Deep Learning tutorial 2](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/JAX/tutorial2/Introduction_to_JAX.html), which is released under the [MIT License](https://github.com/phlippe/uvadlc_notebooks/blob/master/LICENSE.md) (Copyright (c) 2020 Phillip Lippe).

In [ ]:
import os

import numpy as np
import torch
import torch.utils.data as data
from scipy.stats import gaussian_kde
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set()

from src.data.dataset import choice_collate
from src.models.create_model import create_model
from src.models.elbo import log_likelihood
from src.pipelines.unified_training import unified_train_model, compute_forward_loss, move_batch_to_device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
SAMPLE_NEGATIVES = False

PLOT_DIR = "data/plots"
os.makedirs(PLOT_DIR, exist_ok=True)

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

In [ ]:
num_users = 5
num_items = 50
num_options = 10
num_user_types = 5

_np_rng = np.random.RandomState(seed)

init_params = {
    "user_mean": np.array([[2, -2], [2, 2]], dtype=np.float64),
    "user_cov": np.array([[[0.15, 0], [0, 0.15]], [[0.15, 0], [0, 0.15]]], dtype=np.float64),
}

true_params = {
    "item_embs": _np_rng.multivariate_normal(np.zeros(2), np.eye(2), size=num_items),
    "num_options": num_options,
    "num_users": num_users,
}

true_params["user_mean"] = np.stack([
    init_params["user_mean"] @ np.array([
        [np.cos(2 * np.pi * user_id / num_user_types), -np.sin(2 * np.pi * user_id / num_user_types)],
        [np.sin(2 * np.pi * user_id / num_user_types), np.cos(2 * np.pi * user_id / num_user_types)],
    ])
    for user_id in range(num_users)
])

true_params["user_cov"] = np.stack([
    init_params["user_cov"]
    for user_id in range(num_users)
])

In [ ]:
colors = ["#2b474b", "#00e64d", "#0066ff", "#ff6600", "#cc0000"]
fig = plt.figure()
ax = plt.gca()
for i in range(num_users):
    user_mean = true_params["user_mean"][i]
    ax.scatter(user_mean[:,0], user_mean[:,1], color=colors[i], label=f"User {i}")
plt.scatter(true_params["item_embs"][:,0], true_params["item_embs"][:,1], color="red", alpha=0.3, label="Items", marker="x")
ax.legend()
ax.grid(False)
ax.axhline(0, color="black", lw=0.5)
ax.axvline(0, color="black", lw=0.5)

fig.savefig(os.path.join(PLOT_DIR, "Synthetic_true_embedding_plot.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
def draw_true_user_embs(user_id, params, num_samples):
    embs = []
    for user_mean, user_cov in zip(params["user_mean"][user_id], params["user_cov"][user_id]):
        embs.append(np.random.multivariate_normal(user_mean, user_cov, num_samples))

    inds = np.random.choice(len(embs), size=(num_samples,), replace=True)
    embs = [embs[i][idx] for idx, i in enumerate(inds)]
    return np.stack(embs)


user_embs = draw_true_user_embs(params=true_params, user_id=0, num_samples=1000)

plt.figure(figsize=(4, 4))
sns.kdeplot(x=user_embs[:, 0], y=user_embs[:, 1], fill=True, cmap="viridis")
plt.xlabel("User embedding 1")
plt.ylabel("User embedding 2")
plt.title("True density of user 0")
plt.show()

In [ ]:
true_item_embs = true_params["item_embs"]

true_u_embs_plt = {}
for user_id in range(num_users):
    true_u_embs_plt[user_id] = draw_true_user_embs(
        user_id=user_id,
        params=true_params,
        num_samples=1000,
    )

user_embs_sampled = np.vstack(list(true_u_embs_plt.values())).reshape(-1, 2)

fig = plt.figure(figsize=(4, 4))
ax = plt.gca()
ax.axhline(0, color="black", lw=0.5, alpha=0.5)
ax.axvline(0, color="black", lw=0.5, alpha=0.5)

dim1, dim2 = (0, 1)

x_min = min(true_item_embs[:, dim1].min(), user_embs_sampled[:, dim1].min())
x_max = max(true_item_embs[:, dim1].max(), user_embs_sampled[:, dim1].max())
y_min = min(true_item_embs[:, dim2].min(), user_embs_sampled[:, dim2].min())
y_max = max(true_item_embs[:, dim2].max(), user_embs_sampled[:, dim2].max())

x_pad = 0.25 * (x_max - x_min)
y_pad = 0.25 * (y_max - y_min)
x_min -= x_pad; x_max += x_pad; y_min -= y_pad; y_max += y_pad

xi, yi = np.mgrid[x_min:x_max:100j, y_min:y_max:100j]

ax.scatter(true_item_embs[:, 0], true_item_embs[:, 1], label="True Item Embeddings")

for user_id in range(num_users):
    kde_user = gaussian_kde(np.vstack([true_u_embs_plt[user_id][:, dim1], true_u_embs_plt[user_id][:, dim2]]))
    zi_reshaped = kde_user(np.vstack([xi.flatten(), yi.flatten()])).reshape(xi.shape)
    levels_to_use = np.linspace(zi_reshaped.min(), zi_reshaped.max(), 5)[1:]
    contour = ax.contour(xi, yi, zi_reshaped, colors=colors[user_id], alpha=0.5,
                         linewidths=1, linestyles="solid", levels=levels_to_use)
    ax.contourf(xi, yi, zi_reshaped, levels=contour.levels[0:], colors=colors[user_id], alpha=0.15)

ax.set_xlim(-4.2, 4.2)
ax.set_ylim(-4.2, 4.2)
ax.grid()

plt.tight_layout()
plt.show()

In [ ]:
class ChoiceDataset(data.Dataset):

    def __init__(self, size, seed, true_params, sample_negatives=False):
        super().__init__()
        self.size = size
        self.np_rng = np.random.RandomState(seed=seed)
        self.true_params = true_params
        self.sample_negatives = sample_negatives
        self.all_items = set(np.arange(1, len(self.true_params["item_embs"]) + 1))
        self.generate_continuous_choices()

    def generate_continuous_choices(self):
        self.user_ids = self.np_rng.choice(
            a=np.arange(self.true_params["num_users"]),
            size=(self.size,),
            replace=True,
        )

        embs = [
            [
                self.np_rng.multivariate_normal(user_mean, user_cov, size=1)
                for user_mean, user_cov in zip(self.true_params["user_mean"][user_id], self.true_params["user_cov"][user_id])
            ]
            for user_id in self.user_ids
        ]

        self.user_embs = np.asarray(embs)

        user_emb_mix_idxs = self.np_rng.choice(
            a=np.arange(len(self.true_params["user_mean"][0])),
            size=(self.size,),
            replace=True,
        )

        self.user_embs = self.user_embs[np.arange(self.size), user_emb_mix_idxs]

        self.all_item_embs = np.concatenate([np.zeros((1, self.true_params["item_embs"].shape[1])), self.true_params["item_embs"]], axis=0)

        self.option_ids = 1 + self.np_rng.choice(len(self.true_params["item_embs"]), size=(self.size, self.true_params["num_options"]))
        self.option_ids = np.concatenate([np.zeros((self.option_ids.shape[0], 1)).astype(int), self.option_ids], axis=1)

        self.item_embs = self.all_item_embs[self.option_ids]

        det_utils = np.sum(np.multiply(self.user_embs, self.item_embs), axis=-1)
        rand_utils = det_utils + self.np_rng.gumbel(loc=0, scale=1, size=det_utils.shape)
        self.choice_pos = np.argmax(rand_utils, axis=-1)
        self.choice_ids = self.option_ids[np.arange(self.size), self.choice_pos]

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        user_id = self.user_ids[idx]
        option_id = self.option_ids[idx]
        option_emb = self.item_embs[idx]
        choice_pos = self.choice_pos[idx]
        choice_pos_onehot = np.eye(self.true_params["num_options"]+1)[choice_pos]
        choice_id = self.choice_ids[idx]

        if self.sample_negatives:
            available_items = self.all_items - {choice_id}
            option_id = self.np_rng.choice(
                a=np.array(list(available_items)),
                size=option_id.shape,
                replace=False,
            )
            option_id[choice_pos] = choice_id
            option_id[0] = 0
            option_emb = self.all_item_embs[option_id]

        return (
            {
                "user_ids": user_id,
                "option_id": option_id,
                "item_embs": np.asarray(option_emb)
            },
            {
                "choice_pos": choice_pos,
                "choice_pos_onehot": choice_pos_onehot,
                "choice_id": choice_id,
            }
        )

In [ ]:
model_name = "MXL_Corr"

hyperparams = {
    "batch_size": 2 * 512,
    "num_hidden": 16,
    "learning_rate": 0.01,
    "num_early_stopping": 50,
    "num_layers": 0,
    "out_dim": 2,
    "latent_dim": 2,
    "block_length": 2,
    "num_prior_samples": 32,
    "beta_max": 0.,
    "has_intercept": False,
    "l2_weight": 0.001,
    "l2_weight_layers": 0,
    "dropout_rate": 0.0,
    "cyclic_beta_schedule": 20,
    "use_output_layer_norm": False,
    "dist_fn": "dot",
    "cov_type": "aat",
    "cov_rank": 2,
}

model = create_model(
    model_name=model_name,
    num_users=num_users,
    num_items=num_items + 1,
    hyperparams=hyperparams,
    no_choice_id=0,
    padding_id=None,
    never_chosen_id=None,
)
model = model.to(device)

In [ ]:
train_dataset = ChoiceDataset(size=10000, seed=42, true_params=true_params, sample_negatives=SAMPLE_NEGATIVES)
train_data_loader = data.DataLoader(train_dataset, batch_size=hyperparams["batch_size"], shuffle=True, collate_fn=choice_collate)

optimizer = torch.optim.Adam(model.parameters(), lr=hyperparams["learning_rate"])

In [ ]:
batch = next(iter(train_data_loader))
batch = move_batch_to_device(batch, device)

model.eval()
with torch.no_grad():
    logits, probs = model(batch[0])
    components, _ = compute_forward_loss(
        model, model_name, batch[0], batch[1]["choice_pos_onehot"],
        beta=model.beta_max,
        l2_weight=model.l2_weight,
        return_component_values=True,
    )
print(f"NLL of the untrained model on the batch: {float(-components[1].mean()):.4f}")

In [ ]:
trained_model, _ = unified_train_model(
    model=model,
    model_name=model_name,
    optimizer=optimizer,
    train_data_loader=train_data_loader,
    val_data_loader=None,
    num_epochs=201,
    num_early_stopping=hyperparams["num_early_stopping"],
    results_dir="data/results/synthetic_example",
    device=device,
)

In [ ]:
with torch.no_grad():
    components, _ = compute_forward_loss(
        trained_model, model_name, batch[0], batch[1]["choice_pos_onehot"],
        beta=trained_model.beta_max,
        l2_weight=trained_model.l2_weight,
        return_component_values=True,
    )
print(f"NLL of the trained model on the batch: {float(-components[1].mean()):.4f}")

In [ ]:
est_item_embs = trained_model.item_embs.detach().cpu().numpy()

trained_model.eval()
with torch.no_grad():
    est_user_embs = trained_model.get_all_user_out_vectors(num_samples_per_user=32).detach().cpu().numpy()

In [ ]:
trained_model.eval()
true_user_samples = {}
for user_id in [0, 2]:
    with torch.no_grad():
        est_emb_this_user = trained_model.draw_user_embedding_sample(
            user_ids=torch.full((512,), user_id, dtype=torch.long, device=device),
            num_samples=512,
            num_prior_samples=1,
        ).squeeze().detach().cpu().numpy()

    true_user_samples[user_id] = draw_true_user_embs(user_id=user_id, params=true_params, num_samples=trained_model.num_prior_samples)

    plt.scatter(true_user_samples[user_id][:, 0], true_user_samples[user_id][:, 1], s=0.4, label="True")
    plt.scatter(est_emb_this_user[:, 0], est_emb_this_user[:, 1], s=0.4, label="Estimated")

    ax = plt.gca()
    ax.set_xlim(min(true_user_samples[user_id][:, 0].min(), est_emb_this_user[:, 0].min()) - 1, max(true_user_samples[user_id][:, 0].max(), est_emb_this_user[:, 0].max()) + 1)
    ax.set_ylim(min(true_user_samples[user_id][:, 1].min(), est_emb_this_user[:, 1].min()) - 1, max(true_user_samples[user_id][:, 1].max(), est_emb_this_user[:, 1].max()) + 1)
    ax.legend(markerscale=10)
    plt.title(f"User {user_id}: true and estimated samples")
    plt.show()

In [ ]:
for user_id in [0, 2]:
    with torch.no_grad():
        user_emb_samples = trained_model.draw_user_embedding_sample(
            user_ids=torch.full((512,), user_id, dtype=torch.long, device=device),
            num_samples=512,
            num_prior_samples=1,
        ).reshape(-1, trained_model.out_dim).detach().cpu().numpy()

    plt.figure(figsize=(3, 3))
    sns.kdeplot(x=user_emb_samples[:, 0], y=user_emb_samples[:, 1], fill=True, cmap="viridis")
    plt.xlabel("User embedding 1")
    plt.ylabel("User embedding 2")
    plt.title(f"Estimated density of user {user_id}")
    plt.show()

In [ ]:
true_item_vector_std = np.std(true_item_embs, axis=0)
true_user_vector_std = np.std(true_user_samples[2], axis=0)
item_vector_std = np.std(est_item_embs, axis=0)
user_vector_std = np.std(est_user_embs.reshape(-1, est_user_embs.shape[-1]), axis=0)

est_user_embs_scaled = est_user_embs * (true_user_vector_std / user_vector_std)
item_embs_scaled = est_item_embs * (true_item_vector_std / item_vector_std)

In [ ]:
user_embs_sampled = est_user_embs_scaled.reshape(-1, 2)

fig = plt.figure(figsize=(4, 4))
ax = plt.gca()
ax.axhline(0, color="black", lw=0.5, alpha=0.5)
ax.axvline(0, color="black", lw=0.5, alpha=0.5)

dim1, dim2 = (0, 1)

x_min = user_embs_sampled[:, dim1].min()
x_max = user_embs_sampled[:, dim1].max()
y_min = user_embs_sampled[:, dim2].min()
y_max = user_embs_sampled[:, dim2].max()

x_pad = 0.25 * (x_max - x_min)
y_pad = 0.25 * (y_max - y_min)
x_min -= x_pad; x_max += x_pad; y_min -= y_pad; y_max += y_pad

xi, yi = np.mgrid[x_min:x_max:100j, y_min:y_max:100j]

ax.scatter(item_embs_scaled[:, 0], item_embs_scaled[:, 1], label="Est. Item Embeddings")

for user_id in range(num_users):
    kde_user = gaussian_kde(np.vstack([est_user_embs_scaled[user_id][:, dim1], est_user_embs_scaled[user_id][:, dim2]]))
    zi_reshaped = kde_user(np.vstack([xi.flatten(), yi.flatten()])).reshape(xi.shape)
    levels_to_use = np.linspace(zi_reshaped.min(), zi_reshaped.max(), 5)[1:]
    contour = ax.contour(xi, yi, zi_reshaped, colors=colors[user_id], alpha=0.5,
                         linewidths=1, linestyles="solid", levels=levels_to_use)
    ax.contourf(xi, yi, zi_reshaped, levels=contour.levels[0:], colors=colors[user_id], alpha=0.15)

ax.grid()
ax.set_xlim(-4.2, 4.2)
ax.set_ylim(-4.2, 4.2)

plt.tight_layout()
if SAMPLE_NEGATIVES:
    fig.savefig(os.path.join(PLOT_DIR, "Synthetic_estimated_embedding_plot_sampled_negatives.pdf"), bbox_inches="tight")
else:
    fig.savefig(os.path.join(PLOT_DIR, "Synthetic_estimated_embedding_plot_hard_negatives.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
test_dataset = ChoiceDataset(size=10000, seed=55, true_params=true_params, sample_negatives=False)
test_data_loader = data.DataLoader(test_dataset, batch_size=hyperparams["batch_size"], shuffle=True, collate_fn=choice_collate)

user_embs_test = {}
for user_id in range(num_users):
    user_embs_test[user_id] = draw_true_user_embs(user_id=user_id, params=true_params, num_samples=1280)

In [ ]:
est_nlls = []
oracle_nlls = []
with torch.no_grad():
    for batch in tqdm(test_data_loader, total=len(test_data_loader), ascii=True, leave=False):
        batch = move_batch_to_device(batch, device)

        components, _ = compute_forward_loss(
            trained_model, model_name, batch[0], batch[1]["choice_pos_onehot"],
            beta=trained_model.beta_max,
            l2_weight=trained_model.l2_weight,
            return_component_values=True,
        )
        est_nlls.append(float(-components[1].mean()))

        item_embs_np = batch[0]["item_embs"].cpu().numpy()
        onehot = batch[1]["choice_pos_onehot"].cpu()
        user_ids_np = batch[0]["user_ids"].cpu().numpy()

        true_logits = []
        for idx, user_id in enumerate(user_ids_np):
            ue = user_embs_test[int(user_id)]
            true_logits.append(np.sum(ue[:, None, :] * item_embs_np[idx], axis=-1))
        true_logits = np.asarray(true_logits)
        true_probs = np.exp(true_logits) / np.sum(np.exp(true_logits), axis=-1, keepdims=True)
        true_probs = np.mean(true_probs, axis=1)

        oracle_nlls.append(float(-log_likelihood(
            probs=torch.as_tensor(true_probs, dtype=torch.float32),
            labels=onehot.float(),
            logits=torch.as_tensor(true_logits.transpose(0, 2, 1), dtype=torch.float32),
        ).mean()))

print(f"Estimated NLL: {np.mean(est_nlls):.4f}")
print(f"Oracle NLL:    {np.mean(oracle_nlls):.4f}")